<a href="https://colab.research.google.com/github/viictorsauraa/AprendizajeporRefuerzo_SauraCarmonaCortesGrupo7/blob/main/Entornos_Complejos/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aprendizaje en Entornos Complejos

## Descripción

Este notebook es el punto de entrada para el estudio del **aprendizaje por refuerzo en entornos complejos**.
Se estudian dos grupos de técnicas aplicadas sobre entornos de [Gymnasium](https://gymnasium.farama.org/):

- **Parte 2 — Métodos Tabulares**: Monte Carlo, SARSA y Q-Learning sobre los entornos discretos **Taxi-v3** y **CliffWalking-v0**.
- **Parte 3 — Control con Aproximaciones**: SARSA semi-gradiente y Deep Q-Learning (DQN) sobre el entorno continuo **LunarLander-v3**.

LunarLander-v3 tiene un espacio de observación continuo de 8 variables reales, lo que hace **imposible** usar una
tabla Q directamente y justifica el uso de redes neuronales como aproximadores de la función de valor.

### Entornos utilizados

| Entorno | Tipo | Observación | Acciones | Uso |
|---|---|---|---|---|
| **Taxi-v3** | Discreto | 500 estados | 6 (N, S, E, O, recoger, dejar) | Métodos tabulares |
| **CliffWalking-v0** | Discreto | 48 estados | 4 (N, S, E, O) | Comparativa SARSA vs Q-Learning |
| **LunarLander-v3** | Continuo | 8 variables reales | 4 (nada, izq, motor, dcha) | Métodos aproximados |


## Preparación del entorno


In [ ]:
#@title Clonar el repositorio e instalar dependencias
import os

#@title Clonar el repositorio e instalar dependencias

if not os.path.exists('/content/AprendizajeporRefuerzo_SauraCarmonaCortesGrupo7'):
    !git clone https://github.com/viictorsauraa/AprendizajeporRefuerzo_SauraCarmonaCortesGrupo7.git /content/AprendizajeporRefuerzo_SauraCarmonaCortesGrupo7
else:
    print("Repositorio ya clonado")

%cd /content/AprendizajeporRefuerzo_SauraCarmonaCortesGrupo7/Entornos_Complejos

!pip install -q gymnasium[box2d] torch matplotlib tqdm
print("Entorno listo.")

In [ ]:
#@title Verificar instalación
import sys
sys.path.insert(0, "src")

import numpy as np
import gymnasium as gym
import torch

from agents import (
    MonteCarloOnPolicyAgent, MonteCarloOffPolicyAgent,
    SARSAAgent, QLearningAgent,
    SARSASGAgent, DQNAgent
)

print("Todos los agentes importados correctamente.")
print(f"PyTorch: {torch.__version__} | Gymnasium: {gym.__version__}")

## Estructura del estudio

---

### Parte 2 — Métodos Tabulares (Taxi-v3)


#### [MonteCarloTodasLasVisitas.ipynb](https://colab.research.google.com/github/viictorsauraa/AprendizajeporRefuerzo_SauraCarmonaCortesGrupo7/blob/main/Entornos_Complejos/MonteCarloTodasLasVisitas.ipynb)
Notebook de referencia: **MC On-Policy todas las visitas** (Algoritmo 3 de Sutton & Barto) sobre el entorno **FrozenLake-v1** (4×4 y 8×8, sin deslizamiento). Permite estudiar el comportamiento del agente en un entorno sencillo antes de pasar a Taxi-v3.

#### [MonteCarlo_experiment.ipynb](https://colab.research.google.com/github/viictorsauraa/AprendizajeporRefuerzo_SauraCarmonaCortesGrupo7/blob/main/Entornos_Complejos/MonteCarlo_experiment.ipynb)
Comparativa de **Monte Carlo On-Policy** (todas las visitas, Algoritmo 3 de Sutton & Barto) y **Monte Carlo Off-Policy** (muestreo por importancia ponderado, Algoritmo 6) sobre Taxi-v3. Se analiza el efecto de epsilon, el decaimiento y la inicialización de Q.

#### [MonteCarloOffPolicy.ipynb](https://colab.research.google.com/github/viictorsauraa/AprendizajeporRefuerzo_SauraCarmonaCortesGrupo7/blob/main/Entornos_Complejos/MonteCarloOffPolicy.ipynb)
Estudio en profundidad del algoritmo **MC Off-Policy** con importancia ponderada: política de comportamiento vs. política objetivo, efecto del ratio de importancia W y convergencia.

#### [SARSA_experiment.ipynb](https://colab.research.google.com/github/viictorsauraa/AprendizajeporRefuerzo_SauraCarmonaCortesGrupo7/blob/main/Entornos_Complejos/SARSA_experiment.ipynb)
**SARSA** (State-Action-Reward-State-Action): algoritmo TD on-policy tabular sobre Taxi-v3. Se estudia el efecto del learning rate alpha, gamma y epsilon.

#### [SARSA_experiment_CLIFF.ipynb](https://colab.research.google.com/github/viictorsauraa/AprendizajeporRefuerzo_SauraCarmonaCortesGrupo7/blob/main/Entornos_Complejos/SARSA_experiment_CLIFF.ipynb)
**SARSA en CliffWalking-v0**: mismo algoritmo aplicado al entorno del acantilado, donde se ilustra su comportamiento conservador al mantenerse alejado del borde por su naturaleza on-policy.

#### [Qlearning_experiment.ipynb](https://colab.research.google.com/github/viictorsauraa/AprendizajeporRefuerzo_SauraCarmonaCortesGrupo7/blob/main/Entornos_Complejos/Qlearning_experiment.ipynb)
**Q-Learning**: algoritmo TD off-policy tabular sobre Taxi-v3. Comparado con SARSA, usa el valor máximo del siguiente estado independientemente de la política seguida.

#### [Qlearning_experiment_CLIFF.ipynb](https://colab.research.google.com/github/viictorsauraa/AprendizajeporRefuerzo_SauraCarmonaCortesGrupo7/blob/main/Entornos_Complejos/Qlearning_experiment_CLIFF.ipynb)
**Q-Learning en CliffWalking-v0**: comparativa directa con SARSA en el mismo entorno. Q-Learning aprende la ruta óptima (borde del acantilado) pero durante el entrenamiento acumula más penalizaciones por su política de comportamiento ε-greedy.

---

### Parte 3 — Control con Aproximaciones (LunarLander-v3)

#### [SARSA_SG_experiment.ipynb](https://colab.research.google.com/github/viictorsauraa/AprendizajeporRefuerzo_SauraCarmonaCortesGrupo7/blob/main/Entornos_Complejos/SARSA_SG_experiment.ipynb)
**SARSA semi-gradiente**: extiende SARSA tabular usando una red neuronal como aproximador de Q(s,a;w). El target se trata como constante (). Se estudia el efecto de la arquitectura (capas, neuronas) y el learning rate.

#### [DeepQLearning_experiment.ipynb](https://colab.research.google.com/github/viictorsauraa/AprendizajeporRefuerzo_SauraCarmonaCortesGrupo7/blob/main/Entornos_Complejos/DeepQLearning_experiment.ipynb)
**Deep Q-Learning (DQN)**: implementa las mejoras de Mnih et al. (2015) sobre Q-Learning con redes neuronales — **Replay Buffer** (rompe correlaciones temporales) y **Target Network** (estabiliza el target). Se estudia el efecto del tamaño del buffer, la frecuencia de actualización de la red objetivo y la arquitectura.

---

## Agentes implementados

| Agente | Clase | Tipo | Parámetros clave |
|---|---|---|---|
| MC On-Policy |  | Tabular | ,  |
| MC Off-Policy |  | Tabular | , ,  |
| SARSA |  | Tabular | , ,  |
| Q-Learning |  | Tabular | , ,  |
| SARSA semi-gradiente |  | Aproximado | , ,  |
| Deep Q-Learning |  | Aproximado | , , ,  |
